<a href="https://colab.research.google.com/github/budennovsk/AuthorBooksComments/blob/master/v8_2_level_model_rerank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install implicit catboost rectools[lightfm] gensim optuna



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import typing as tp
import warnings

import pandas as pd
import numpy as np

from implicit.nearest_neighbours import CosineRecommender
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifier
from catboost import CatBoostClassifier, CatBoostRanker
try:
    from lightgbm import LGBMClassifier, LGBMRanker
    LGBM_AVAILABLE = True
except ImportError:
    warnings.warn("lightgbm is not installed. Some parts of the notebook will be skipped.")
    LGBM_AVAILABLE = False
from rectools.dataset import Interactions
from lightfm import LightFM
from rectools import Columns
from rectools.dataset import Dataset
from rectools.metrics import Precision, Recall, MeanInvUserFreq, Serendipity,MAP,NDCG,HitRate,calc_metrics
from rectools.models import (
    ImplicitALSWrapperModel,
    ImplicitBPRWrapperModel,
    LightFMWrapperModel,
    PureSVDModel,
    ImplicitItemKNNWrapperModel,
    EASEModel,
    RandomModel,
    PopularInCategoryModel,
    PopularModel,
    EASEModel, SASRecModel, BERT4RecModel)
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking
from implicit.nearest_neighbours import CosineRecommender
from rectools.models.base import ExternalIds
from rectools.models.ranking import (
    CandidateRankingModel,
    CandidateGenerator,
    Reranker,

    CatBoostReranker,
    CandidateFeatureCollector,
    PerUserNegativeSampler
)
from rectools.models.nn.item_net import CatFeaturesItemNet, IdEmbeddingsItemNet
from rectools.model_selection import cross_validate, TimeRangeSplitter,LastNSplitter,Splitter
from tqdm.auto import tqdm
import datetime
import dill
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from zipfile import ZipFile
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

In [ ]:
path_users = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/users.csv'
path_items = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/items.csv'
path_interactions = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/interactions.csv'


users_all = pd.read_csv(path_users)
items_all = pd.read_csv(path_items)
interactions = (
    pd.read_csv(path_interactions, parse_dates=["last_watch_dt"])
    .rename(columns={"last_watch_dt": Columns.Datetime})
)

# interactions["last_watch_dt"] = pd.to_datetime(interactions["last_watch_dt"], errors="coerce")
interactions = interactions.head(1000000)
users = users_all[users_all['user_id'].isin(interactions['user_id'].unique())]
items = items_all[items_all['item_id'].isin(interactions['item_id'].unique())]
interactions["weight"] = 1
dataset = Dataset.construct(interactions)
# RANDOM_STATE = 32


# dataset
# interactions[Columns.Datetime] = pd.to_datetime(interactions[Columns.Datetime], format='%Y-%m-%d')

In [ ]:
ratings = interactions[['user_id', 'item_id']]
grouped_df = ratings.groupby("user_id").agg({
    "item_id": lambda x: list(x),
}).reset_index()

# Разделяем на train/test
grouped_df["train_item_ids"] = grouped_df["item_id"].apply(lambda x: x[:-3])
grouped_df["test_item_ids"] = grouped_df["item_id"].apply(lambda x: x[-3:])
# grouped_df["train_ratings"] = grouped_df["rating"].apply(lambda x: x[:-5])
# grouped_df["test_ratings"] = grouped_df["rating"].apply(lambda x: x[-5:])

# Оставляем только нужные колонки
grouped_df = grouped_df[["user_id", "train_item_ids", "test_item_ids"]]

grouped_df["user_id"] = range(len(grouped_df))
grouped_df = grouped_df[grouped_df['train_item_ids'].apply(len) > 0].reset_index(drop=True)

grouped_df



In [ ]:
# Для pandas
median_seq_len = grouped_df['train_item_ids'].apply(len).median()
median_seq_len = int(median_seq_len)
print(f"средняя длина сессии {median_seq_len}")

In [ ]:
from typing import List, Any
def map_at_k(y_true: List[int], y_pred: List[int], k: int = 20) -> float:
    """
    MAP@k (Mean Average Precision)
    y_true: [1566, 1907, 48] - релевантные товары
    y_pred: [356, 1073, 3052, ...] - предсказанные товары
    """
    y_true_set = set(y_true)
    y_pred_k = y_pred[:k]

    # if not y_true_set:
    #     return 0.0

    score = 0.0
    hits = 0
    for i, item in enumerate(y_pred_k, start=1):
        if item in y_true_set:
            hits += 1
            score += hits / i

    return score / min(len(y_true_set), k)


def ndcg_at_k(y_true: List[int], y_pred: List[int], k: int = 20) -> float:
    """
    NDCG@k (Normalized Discounted Cumulative Gain)
    y_true: [1566, 1907, 48] - релевантные товары
    y_pred: [356, 1073, 3052, ...] - предсказанные товары
    """
    y_true_set = set(y_true)
    y_pred_k = y_pred[:k]

    # if not y_true_set:
    #     return 0.0

    # DCG - реальный дисконтированный выигрыш
    dcg = 0.0
    for i, item in enumerate(y_pred_k, start=1):
        if item in y_true_set:
            dcg += 1.0 / np.log2(i + 1)

    # IDCG - идеальный дисконтированный выигрыш
    ideal_hits = min(len(y_true_set), k)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))

    return dcg / idcg if idcg > 0 else 0.0


def hitrate_at_k(y_true: List[int], y_pred: List[int], k: int = 20) -> int:
    """
    Hitrate@k (HR@k) - попал ли хотя бы один релевантный товар в топ-k?
    y_true: [1566, 1907, 48] - релевантные товары
    y_pred: [356, 1073, 3052, ...] - предсказанные товары
    Возвращает: 1 если попал, 0 если нет
    """
    y_true_set = set(y_true)
    y_pred_k = y_pred[:k]

    return int(len(y_true_set.intersection(y_pred_k)) > 0)

In [ ]:
import optuna


def evaluate_model(model):
    ndcg_list = []
    hitrate_list = []
    map_list = []



    for train_ids, y_rel in zip(grouped_df['train_item_ids'], grouped_df['test_item_ids']):
        context = [x for x in train_ids[-model.window:] if x in model.wv]
        if not context:
            continue

        model_preds = model.predict_output_word(
            train_ids[-model.window :], topn=(TOP_K + len(train_ids))
    )

        # if model_preds is None:
        #     ndcg_list.append(0)
        #     hitrate_list.append(0)
        #     continue
        # context = [x for x in train_ids[-model.window:] if x in model.wv]

        # if not context:
        #     continue  # Пропускаем если нет контекста
        y_rec = [int(pred[0]) for pred in model_preds if int(pred[0]) not in train_ids]
        y_rel = [int(x) for x in y_rel]

        ndcg_list.append(ndcg_at_k(y_rel, y_rec, k=20))
        hitrate_list.append(hitrate_at_k(y_rel, y_rec, k=20))
        map_list.append(map_at_k(y_rel, y_rec, k=20))
    return np.mean(ndcg_list), np.mean(hitrate_list), np.mean(map_list)


def objective(trial):
    sg = trial.suggest_categorical("sg", [0, 1])
    window = trial.suggest_int("window", 1, 10)
    ns_exponent = trial.suggest_float("ns_exponent", -3, 3)
    negative = trial.suggest_int("negative", 3, 20)
    min_count = trial.suggest_int("min_count", 0, 20)
    vector_size = trial.suggest_categorical("vector_size", [16, 32, 64, 128])

    print(
        {
            "sg": sg,
            "window_len": window,
            "ns_exponent": ns_exponent,
            "negative": negative,
            "min_count": min_count,
            "vector_size": vector_size,
        }
    )

    set_seed()
    model = Word2Vec(
        grouped_df["train_item_ids"].to_list(),
        window=window,
        sg=sg,
        hs=0,
        min_count=min_count,
        vector_size=vector_size,
        negative=negative,
        ns_exponent=ns_exponent,
        seed=RANDOM_STATE,
        epochs=10,
    )

    mean_ndcg, mean_hitrate,map_mean = evaluate_model(model)
    print(f"NDCG@{TOP_K} = {mean_ndcg:.4f}, Hitrate@{TOP_K} = {mean_hitrate:.4f} map{TOP_K} = {map_mean:.4f}")
    return mean_ndcg


study = optuna.create_study(directions=("maximize",))
study.optimize(objective, n_trials=20)

study.best_params

In [ ]:
from gensim.models import Word2Vec
import numpy as np
import pandas as pd
import random

RANDOM_STATE = 42
def set_seed():
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)



TOP_K=20
def evaluate_model(model):
    ndcg_list = []
    hitrate_list = []
    map_list = []
    y_rel_list = []
    y_rec_list = []

    for train_ids, y_rel in zip(grouped_df['train_item_ids'], grouped_df['test_item_ids']):
        context = [x for x in train_ids[-model.window:] if x in model.wv]

        if not context:
            continue  # Пропускаем если нет контекста

        model_preds = model.predict_output_word(
            train_ids[-model.window :], topn=(TOP_K + len(train_ids))
    )
        # print(train_ids)
        # print(y_rel)
        # print(train_ids[-model.window :])
        # 22/0
        # context = [x for x in train_ids[-model.window:] if x in model.wv]

        # if not context:
        #     continue  # Пропускаем если нет контекста

        model_preds = model.predict_output_word(
            train_ids[-model.window :], topn=(TOP_K + len(train_ids))
        )

        # if model_preds is None:
        #     ndcg_list.append(0)
        #     hitrate_list.append(0)
        #     map_list.append(0)
        #     y_rel_list.append(0)
        #     y_rec_list.append(0)
        #     continue

        y_rec = [int(pred[0]) for pred in model_preds if int(pred[0]) not in train_ids]
        y_rel = [int(x) for x in y_rel]
        # Подсчитываем метрики для k=20
        ndcg_list.append(ndcg_at_k(y_rel, y_rec, k=20))
        hitrate_list.append(hitrate_at_k(y_rel, y_rec, k=20))
        map_list.append(map_at_k(y_rel, y_rec, k=20))

        y_rel_list.append(y_rel)
        y_rec_list.append(y_rec)

    # Создаем дата��рейм для визуального сравнения
    results_df = pd.DataFrame({
        'y_rel': y_rel_list,
        'y_rec': y_rec_list,
        'ndcg': ndcg_list,
        'hr': hitrate_list,
        'map': map_list
    })

    return results_df, np.mean(ndcg_list), np.mean(hitrate_list), np.mean(map_list)


set_seed()
w2v_model = Word2Vec(
    grouped_df['train_item_ids'].to_list(),
    hs=0,
    seed=RANDOM_STATE,
    epochs=1,
    sg=1,
    window=4,
    ns_exponent=-0.5850122781145808,
    negative=8,
    min_count=16,
    vector_size=128,
)

results_df, mean_ndcg, mean_hitrate, mean_map = evaluate_model(w2v_model)

print(f'NDCG@20 = {mean_ndcg:.4f}, HR@20 = {mean_hitrate:.4f}, MAP@20 = {mean_map:.4f}')
results_df

In [ ]:
print(w2v_model)

In [ ]:
from umap import UMAP
import matplotlib.pyplot as plt

# 1. Получаем эмбеdinги из модели
embeddings = w2v_model.wv.vectors  # матрица (n_items, 16)
item_ids = w2v_model.wv.index_to_key  # названия товаров

# 2. Снижаем размерность до 2D с помощью UMAP
umap_model = UMAP(n_components=2, random_state=RANDOM_STATE)
embeddings_2d = umap_model.fit_transform(embeddings)

# 3. Строим график
plt.figure(figsize=(12, 8))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.5, s=20)

# Добавляем названия товаров (опционально, если не слишком много)
if len(item_ids) < 500:
    for i, item_id in enumerate(item_ids):
        plt.annotate(item_id, (embeddings_2d[i, 0], embeddings_2d[i, 1]), fontsize=6)

plt.title('Word2Vec Embeddings (UMAP projection)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from gensim.models import Word2Vec
import numpy as np
import pandas as pd
import random
from collections import Counter

RANDOM_STATE = 42
def set_seed():
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

TOP_K = 20

def calculate_novelty(y_rec, all_items_counter, top_k=20):
    """Новизна - насколько рекомендации редкие"""
    if not y_rec:
        return 0

    total_recs = sum(all_items_counter.values())
    novelty = 0

    for item in y_rec[:top_k]:
        novelty += 1 - (all_items_counter.get(item, 0) / total_recs)

    return novelty / min(top_k, len(y_rec))


def calculate_diversity(y_rec, model, top_k=20):
    """Разнообразие - насколько разные рекомендации между собой"""
    y_rec = [str(x) for x in y_rec[:top_k]]

    if len(y_rec) < 2:
        return 0

    distances = []
    for i in range(len(y_rec)):
        for j in range(i + 1, len(y_rec)):
            if y_rec[i] in model.wv and y_rec[j] in model.wv:
                sim = model.wv.similarity(y_rec[i], y_rec[j])
                distances.append(1 - sim)

    return np.mean(distances) if distances else 0


def calculate_serendipity(y_rel, y_rec, top_k=20):
    """Серендипити - неожиданные но полезные рекомендации"""
    y_rel = set(y_rel)
    y_rec = y_rec[:top_k]

    if not y_rec:
        return 0

    unexpected = sum(1 for item in y_rec if item not in y_rel)
    return unexpected / min(top_k, len(y_rec))


def evaluate_model(model):
    ndcg_list = []
    hitrate_list = []
    map_list = []
    novelty_list = []
    diversity_list = []
    serendipity_list = []
    y_rel_list = []
    y_rec_list = []

    # Первый проход - собираем все рекомендации и метрики
    for train_ids, y_rel in zip(grouped_df['train_item_ids'], grouped_df['test_item_ids']):
        context = [x for x in train_ids[-model.window:] if x in model.wv]

        if not context:
            continue

        model_preds = model.predict_output_word(
            train_ids[-model.window:], topn=(TOP_K + len(train_ids))
        )

        y_rec = [int(pred[0]) for pred in model_preds if int(pred[0]) not in train_ids]
        y_rel = [int(x) for x in y_rel]

        # Считаем основные метрики
        ndcg_list.append(ndcg_at_k(y_rel, y_rec, k=20))
        hitrate_list.append(hitrate_at_k(y_rel, y_rec, k=20))
        map_list.append(map_at_k(y_rel, y_rec, k=20))

        # Считаем дополнительные метрики (пока без novelty)
        diversity_list.append(calculate_diversity(y_rec, model, top_k=20))
        serendipity_list.append(calculate_serendipity(y_rel, y_rec, top_k=20))

        y_rel_list.append(y_rel)
        y_rec_list.append(y_rec)

    # Второй проход - считаем novelty со всеми собранными товарами
    all_items = []
    for rec_list in y_rec_list:
        all_items.extend(rec_list[:TOP_K])
    all_items_counter = Counter(all_items)

    novelty_list = [calculate_novelty(y_rec, all_items_counter, top_k=20) for y_rec in y_rec_list]

    results_df = pd.DataFrame({
        'y_rel': y_rel_list,
        'y_rec': y_rec_list,
        'ndcg': ndcg_list,
        'hr': hitrate_list,
        'map': map_list,
        'novelty': novelty_list,
        'diversity': diversity_list,
        'serendipity': serendipity_list
    })

    return results_df, {
        'ndcg': np.mean(ndcg_list),
        'hr': np.mean(hitrate_list),
        'map': np.mean(map_list),
        'novelty': np.mean(novelty_list),
        'diversity': np.mean(diversity_list),
        'serendipity': np.mean(serendipity_list)
    }

set_seed()
w2v_model = Word2Vec(
    grouped_df['train_item_ids'].to_list(),
    hs=0,
    seed=RANDOM_STATE,
    epochs=1,
    sg=1,
    window=4,
    ns_exponent=-0.5850122781145808,
    negative=8,
    min_count=16,
    vector_size=128,
)

results_df, metrics = evaluate_model(w2v_model)

print(f'NDCG@20 = {metrics["ndcg"]:.4f}')
print(f'HR@20 = {metrics["hr"]:.4f}')
print(f'MAP@20 = {metrics["map"]:.4f}')
print(f'Novelty@20 = {metrics["novelty"]:.4f}')
print(f'Diversity@20 = {metrics["diversity"]:.4f}')
print(f'Serendipity@20 = {metrics["serendipity"]:.4f}')
print()
results_df

In [ ]:
interactions.info()

In [ ]:
max_date = interactions["datetime"].max()
train = interactions[interactions["datetime"] < max_date - pd.Timedelta(days=7)].copy()
test = interactions[interactions["datetime"] >= max_date - pd.Timedelta(days=7)].copy()

print(f"train: {train.shape}")
print(f"test: {test.shape}")

In [ ]:
# отфильтруем холодных пользователей из теста
cold_users = list(set(test['user_id']) - set(train['user_id']))
len(cold_users)

In [ ]:
lfm_date_threshold = train['datetime'].quantile(q=0.6, interpolation='nearest')
lfm_date_threshold

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# предполагается, что train уже есть и train['last_watch_dt'] уже datetime64[ns]
train = train.copy()
train["datetime"] = pd.to_datetime(train["datetime"], errors="coerce")
train = train.dropna(subset=["datetime"]).copy()

# 1) Группировка по дням + количество строк
daily = (
    train.assign(day=train["datetime"].dt.floor("D"))
         .groupby("day", as_index=False)
         .size()
         .rename(columns={"size": "rows"})
         .sort_values("day")
)

# 2) (опционально) кумулятивная доля — помогает увидеть, где будет quantile(0.6)
daily["cum_rows"] = daily["rows"].cumsum()
daily["cum_share"] = daily["cum_rows"] / daily["rows"].sum()

display(daily.head())
display(daily.tail())

# 3) Порог по квантилю времени (как у тебя)
lfm_date_threshold = train["datetime"].quantile(q=0.6, interpolation="nearest")
print("lfm_date_threshold:", lfm_date_threshold)

# 4) График активности по дням
plt.figure(figsize=(14, 5))
plt.plot(daily["day"], daily["rows"], marker="o", linewidth=1)
plt.axvline(pd.to_datetime(lfm_date_threshold).floor("D"), color="red", linestyle="--", linewidth=2, label="q=0.6 threshold (day)")
plt.title("Количество взаимодействий по дням (train, datetime)")
plt.xlabel("День")
plt.ylabel("Количество строк")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 5) График кумулятивной доли (чтобы понять, где примерно 60%)
plt.figure(figsize=(14, 4))
plt.plot(daily["day"], daily["cum_share"], marker=".", linewidth=1)
plt.axhline(0.6, color="red", linestyle="--", linewidth=2, label="0.6")
plt.axvline(pd.to_datetime(lfm_date_threshold).floor("D"), color="red", linestyle="--", linewidth=2, label="threshold day")
plt.title("Кумулятивная доля строк по дням")
plt.xlabel("День")
plt.ylabel("Cum share")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
lfm_train = train[(train['datetime'] < lfm_date_threshold)]
lfm_pred = train[(train['datetime'] >= lfm_date_threshold)]

print(f"lfm_train: {lfm_train.shape}")
print(f"lfm_pred: {lfm_pred.shape}")

In [ ]:
lfm_pred = lfm_pred[lfm_pred['user_id'].isin(lfm_train['user_id'].unique())]
print(f"lfm_pred: {lfm_pred.shape}")

In [ ]:
items_lfm_train=items[items['item_id'].isin(lfm_train['item_id'])]
items_lfm_train["genre"] = items_lfm_train["genres"].str.split(",")
genre_feature = items_lfm_train[["item_id", "genre"]].explode("genre")
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"
genre_feature_train = genre_feature[genre_feature['id'].isin(train['item_id'])]
import re

col = "value"

def canon(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("–", "-").replace("—", "-")
    x = re.sub(r"\s*-\s*", "-", x)  # "ток - шоу" -> "ток-шоу"
    return x

# 1) нормализуем текст
genre_feature_train[col] = genre_feature_train[col].map(canon)

# 2) склейки (синонимы/варианты)
# mapping = {
#     "советские": "русские",
#     # (по написанию)
#     "токшоу": "ток-шоу",
#     "реалити": "реалити-шоу",
#     "шоу": "телешоу",  # если хочешь объединять "шоу" с "телешоу"
#     "мультфильм": "мультфильмы",
#     "русские мультфильмы": "мультфильмы",  # если хочешь все мультфильмы в одну категорию
#     "западные мультфильмы": "мультфильмы",
#     "детские": "для детей",
#     "для самых маленьких": "для детей",    # если хочешь объединять
# }
mapping = {
    "советские": "русские",
    # (по написанию)
    "токшоу": "ток-шоу",
    "реалити": "реалити-шоу",
    "шоу": "телешоу",  # если хочешь объединять "шоу" с "телешоу"
    "мультфильм": "мультфильмы",
    "русские мультфильмы": "мультфильмы",  # если хочешь все мультфильмы в одну категорию
    "западные мультфильмы": "мультфильмы",
    "детские": "для детей",
    "для самых маленьких": "для детей",
    "исторические":	"историческое",
    "музыка":	"музыкальные",
    "музыка":	"мюзиклы",
    "комиксы":"по комиксам",
    "спорт":"футбол",
    "комедии":	"юмор",
}

genre_feature_train[col] = genre_feature_train[col].replace(mapping)
genre_feature_train[col].nunique()

counts = (
    genre_feature_train
    .groupby('value', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

counts_lt10 = counts[counts["count"] < 10].reset_index(drop=True)

counts_lt10  # все value, которые встречаются меньше 10 раз
rare_values = set(counts_lt10["value"])

genre_feature_train["value"] = genre_feature_train["value"].where(
    ~genre_feature_train["value"].isin(rare_values),
    "other",
)
genre_feature_train[col].nunique()
items_features_train_lfm = genre_feature_train.copy()

In [ ]:
user_lfm_train=users[users['user_id'].isin(lfm_train['user_id'])]
user_lfm_train["sex"] = user_lfm_train["sex"].map({"Ж": 1, "М": 0}).fillna(-1).astype("int8")
age_category = pd.CategoricalDtype(
    categories=[
         "unknown",
        'age_18_24',
        'age_25_34',
        'age_35_44',
        'age_45_54',
        'age_55_64',
        'age_65_inf',
    ],
    ordered=True,
)
user_lfm_train["age"] = (
    user_lfm_train["age"]
    .fillna("unknown")
    .astype(age_category)
)
income_category = pd.CategoricalDtype(
    categories=[
        "unknown",          # добавили
        "income_0_20",
        "income_20_40",
        "income_40_60",
        "income_60_90",
        "income_90_150",
        "income_150_inf",
    ],
    ordered=True,
)

# если income уже строки/категории — заполним nan и приведем тип
user_lfm_train["income"] = (
    user_lfm_train["income"]
    .fillna("unknown")
    .astype(income_category)
)
user_features_frames = []
for feature in ["sex", "age", "income"]:
    feature_frame = user_lfm_train.reindex(columns=[Columns.User, feature])
    feature_frame.columns = ["id", "value"]
    feature_frame["feature"] = feature
    user_features_frames.append(feature_frame)
user_features_train = pd.concat(user_features_frames)
user_features_train_lfm=user_features_train.copy()


In [ ]:
dataset_i_u_features_lfm_train = Dataset.construct(
    interactions_df=lfm_train,
    user_features_df=user_features_train_lfm,
    cat_user_features=["sex", "age", "income"],
    item_features_df=items_features_train_lfm,
    cat_item_features=["genre", "content_type"],
)

In [ ]:
# Take few models to compare
from datetime import timedelta
RANDOM_STATE = 42
models_1_level = {
    # "popular": PopularModel(popularity="n_interactions"),
    # "popular_7d": PopularModel(period=timedelta(days=7)),
    # "random": RandomModel(random_state=RANDOM_STATE),
    # "most_raited": PopularModel(popularity="mean_weight"),
    # "pop_cat": PopularInCategoryModel(category_feature='genre', n_categories=20),
    # "cosine_knn": ImplicitItemKNNWrapperModel(CosineRecommender()),
    # 'iALS':ImplicitALSWrapperModel(
    #       AlternatingLeastSquares(
    #         factors=10,  # latent embeddings size
    #         regularization=0.1,
    #         iterations=10,
    #         alpha=50,  # confidence multiplier for non-zero entries in interactions
    #         random_state=RANDOM_STATE)),
    'LightFM':LightFMWrapperModel(
            LightFM(no_components=32,
                    loss="warp",
                    random_state=RANDOM_STATE)),

}

# We will calculate several classic (precision@k and recall@k) and "beyond accuracy" metrics
metrics = {
    "prec@10": Precision(k=10),
    "recall@10": Recall(k=10),
    "novelty@10": MeanInvUserFreq(k=10),
    # "serendipity@10": Serendipity(k=10),
    "MAP@10": MAP(k=10),
    "NDCG@10": NDCG(k=10),
    "HitRate@10": HitRate(k=10)
}

K_RECS = 10

In [ ]:
results_1_level = []
K_RECOS = 10
RANDOM_STATE = 42
NUM_THREADS =-1
for model_name, model in tqdm(models_1_level.items()):
    model_quality = {'model': model_name}

    model.fit(dataset_i_u_features_lfm_train)
    candidates_level_1 = model.recommend(
        users=lfm_pred[Columns.User].unique(),
        dataset=dataset_i_u_features_lfm_train, #dataset_feature_train train_dataset
        k=K_RECOS,
        filter_viewed=True,
    )

    metric_values = calc_metrics(metrics, candidates_level_1, lfm_pred, lfm_train)

    model_quality.update(metric_values)
    results_1_level.append(model_quality)
df_quality_1_level = pd.DataFrame(results_1_level).T

df_quality_1_level.columns = df_quality_1_level.iloc[0]

df_quality_1_level.drop('model', inplace=True)
# df_quality_1_level.style.highlight_max(color='lightgreen', axis=1)
df_quality_1_level

In [ ]:
candidates_level_1['item_id'].nunique()

In [ ]:
#катбуст

In [ ]:
pos = candidates_level_1.merge(lfm_pred,
                        on=['user_id', 'item_id'],
                        how='inner')

pos['target'] = 1
print(pos.shape)
pos.head()

In [ ]:
neg = candidates_level_1.set_index(['user_id', 'item_id'])\
        .join(lfm_pred.set_index(['user_id', 'item_id']))

neg = neg[neg['watched_pct'].isnull()].reset_index()

neg = neg.sample(frac=0.07)
neg['target'] = 0

neg.shape

In [ ]:
ctb_train_users, ctb_test_users = train_test_split(lfm_pred['user_id'].unique(),
                                                  random_state=1,
                                                  test_size=0.2)

In [ ]:
# выделяем 10% под механизм early stopping
ctb_train_users, ctb_eval_users = train_test_split(ctb_train_users,
                                                  random_state=1,
                                                  test_size=0.1)

In [ ]:
select_col = ['user_id', 'item_id', 'rank', 'target']

# Catboost train
ctb_train = shuffle(
    pd.concat([
        pos[pos['user_id'].isin(ctb_train_users)],
        neg[neg['user_id'].isin(ctb_train_users)]
])[select_col]
)

# Catboost test
ctb_test = shuffle(
    pd.concat([
        pos[pos['user_id'].isin(ctb_test_users)],
        neg[neg['user_id'].isin(ctb_test_users)]
])[select_col]
)

# for early stopping
ctb_eval = shuffle(
    pd.concat([
        pos[pos['user_id'].isin(ctb_eval_users)],
        neg[neg['user_id'].isin(ctb_eval_users)]
])[select_col]
)

In [ ]:
ctb_train['target'].value_counts(normalize=True)

In [ ]:
user_col = ['user_id', 'age', 'income', 'sex', 'kids_flg']
item_col = ['item_id', 'content_type', 'countries', 'for_kids', 'age_rating', 'studios']

train_feat = ctb_train.merge(users[user_col],
                           on=['user_id'],
                           how='left')\
                        .merge(items[item_col],
                                   on=['item_id'],
                                   how='left')

eval_feat = ctb_eval.merge(users[user_col],
                           on=['user_id'],
                           how='left')\
                        .merge(items[item_col],
                                   on=['item_id'],
                                   how='left')
test_feat = ctb_test.merge(users[user_col],
                           on=['user_id'],
                           how='left')\
                    .merge(items[item_col],
                               on=['item_id'],
                               how='left')

In [ ]:
drop_col = ['user_id', 'item_id']
target_col = ['target']
cat_col = ['age', 'income', 'sex', 'content_type', 'countries', 'studios']
X_train, y_train = train_feat.drop(drop_col + target_col, axis=1), train_feat[target_col]
X_val, y_val = eval_feat.drop(drop_col + target_col, axis=1), eval_feat[target_col]
X_train, y_train = train_feat.drop(drop_col + target_col, axis=1), train_feat[target_col]
X_val, y_val = eval_feat.drop(drop_col + target_col, axis=1), eval_feat[target_col]
# fillna for catboost with the most frequent value
X_train = X_train.fillna(X_train.mode().iloc[0])
X_val = X_val.fillna(X_train.mode().iloc[0])
test_feat = test_feat.fillna(X_train.mode().iloc[0])
# X_train = X_train.fillna('unknown')
# X_val = X_val.fillna('unknown')
# test_feat = test_feat.fillna('unknown')
X_test, y_test = test_feat.drop(drop_col + target_col, axis=1), test_feat['target']

In [ ]:
from catboost import CatBoostClassifier

# параметры для обучения
est_params = {
  'subsample': 0.9,
  'max_depth': 5,
  'n_estimators': 2000,
  'learning_rate': 0.1,
  'thread_count': 20,
  'random_state': 42,
  'verbose': 200,
}

ctb_model = CatBoostClassifier(**est_params)
ctb_model.fit(X_train,
              y_train,
              eval_set=(X_val, y_val),
              early_stopping_rounds=100,
              cat_features=cat_col,
              plot=False)

In [ ]:
import shap
from catboost import Pool

# сэмплируем для shap_values
X_train_subs, _, y_train_subs, __ = train_test_split(X_train, y_train,
                                                     test_size=0.9,
                                                     random_state=42)
# считаем shap_values
shap_values = ctb_model.get_feature_importance(Pool(X_train_subs, y_train_subs,
                                                   cat_features=cat_col), type='ShapValues')

expected_value = shap_values[0, -1]
shap_values = shap_values[:, :-1]
print(X_train.shape,X_train_subs.shape)
plt.title("Важность фичей на train")

shap.summary_plot(
    shap_values,
    X_train_subs
)

In [ ]:
y_pred = ctb_model.predict_proba(X_test)
y_pred_yt = ctb_model.predict(X_test)
from sklearn.metrics import roc_auc_score

f"ROC AUC score = {roc_auc_score(y_test,y_pred[:,1],):.2f}"

In [ ]:

# оставляем только теплых пользователей
test = test[test['user_id'].isin(lfm_train['user_id'].unique())]

In [ ]:
results_lfm_test= []
K_RECOS = 200
RANDOM_STATE = 42
NUM_THREADS =-1
for model_name, model in tqdm(models_1_level.items()):
    model_quality = {'model': model_name}

    model.fit(dataset_i_u_features_lfm_train)
    candidates_lfm_test = model.recommend(
        users=test[Columns.User].unique(),
        dataset=dataset_i_u_features_lfm_train, #dataset_feature_train train_dataset
        k=K_RECOS,
        filter_viewed=True,
    )

    metric_values = calc_metrics(metrics, candidates_lfm_test, test, lfm_train)

    model_quality.update(metric_values)
    results_lfm_test.append(model_quality)
df_quality_1_level = pd.DataFrame(results_lfm_test).T

df_quality_1_level.columns = df_quality_1_level.iloc[0]

df_quality_1_level.drop('model', inplace=True)
# df_quality_1_level.style.highlight_max(color='lightgreen', axis=1)
df_quality_1_level

In [ ]:
lfm_ctb_prediction = candidates_lfm_test.copy()

# фичи для теста
score_feat = lfm_ctb_prediction.merge(users[user_col],
                                   on=['user_id'],
                                   how='left')\
                                .merge(items[item_col],
                                       on=['item_id'],
                                       how='left')

# fillna for catboost with the most frequent value
score_feat = score_feat.fillna(X_train.mode().iloc[0])
score_feat.head()

In [ ]:
# catboost predict_proba
ctb_prediction = ctb_model.predict_proba(score_feat[X_test.columns])

lfm_ctb_prediction['ctb_pred'] = ctb_prediction[:, 1]
lfm_ctb_prediction.head(3)

In [ ]:
candidates_lfm_test.copy().sort_values(
    by=['user_id'], ascending=[True]).head(10)

In [ ]:
# сортируем по скору внутри одного пользователя и проставляем новый ранг
lfm_ctb_prediction = lfm_ctb_prediction.sort_values(
    by=['user_id', 'ctb_pred'], ascending=[True, False])
lfm_ctb_prediction['rank_ctb'] = lfm_ctb_prediction.groupby('user_id').cumcount() + 1
lfm_ctb_prediction.head(10)

In [ ]:
lfm_ctb_prediction.shape,candidates_lfm_test.shape

In [ ]:
# интересно сравнить ранки 1 этапа lightfm и двухэтапной модели
pd.crosstab(lfm_ctb_prediction[lfm_ctb_prediction['rank'] <= 10]['rank'],
            lfm_ctb_prediction[lfm_ctb_prediction['rank_ctb'] <= 10]['rank_ctb'])\
    .style.background_gradient(cmap='spring')

In [ ]:
lfm_ctb_cand = (
    lfm_ctb_prediction[["user_id", "item_id", "score", "rank_ctb"]]
    .rename(columns={"rank_ctb": "rank"})
)

metric_values_lfm_pred = calc_metrics(metrics, candidates_level_1, lfm_pred, lfm_train)
metric_values_lfm_test = calc_metrics(metrics, candidates_lfm_test, test, lfm_train)
metric_values_lfm_cb_bin = calc_metrics(metrics,lfm_ctb_cand, test, lfm_train)

# 1) Собираем словари в один датафрейм (строки = разные прогоны/наборы метрик)
final_res_metrics = pd.DataFrame([
    {"stage": "lfm_pred_level_1", **metric_values_lfm_pred},
    {"stage": "lfm_test_level_1", **metric_values_lfm_test},
    {"stage": "lfm_test_level_2_ctb", **metric_values_lfm_cb_bin},
])

# 2) (опционально) сделаем stage индексом для удобства
final_res_metrics = final_res_metrics.set_index("stage")

final_res_metrics

In [ ]:
from rectools.models.ranking import (
    CandidateRankingModel,
    CandidateGenerator,
    Reranker,
    CatBoostReranker,
    CandidateFeatureCollector,
    PerUserNegativeSampler
)
from rectools.model_selection import cross_validate, TimeRangeSplitter,LastNSplitter,Splitter

In [ ]:
train = interactions[interactions["datetime"] < max_date - pd.Timedelta(days=7)].copy()
test = interactions[interactions["datetime"] >= max_date - pd.Timedelta(days=7)].copy()
dataset_train = Dataset.construct(train)

In [ ]:
import numpy as np
import pandas as pd
RANDOM_STATE = 42

def _ensure_datetime(df: pd.DataFrame, col: str = "datetime") -> pd.DataFrame:
    out = df.copy()
    if col in out.columns and not np.issubdtype(out[col].dtype, np.datetime64):
        out[col] = pd.to_datetime(out[col], errors="coerce")
    return out


def _safe_log1p(s: pd.Series) -> pd.Series:
    return np.log1p(s.astype("float64"))


def _add_quantiles(features: pd.DataFrame, group: pd.core.groupby.generic.SeriesGroupBy, prefix: str):
    # добавляет q05/q50/q95 как отдельные колонки
    q = group.quantile([0.05, 0.5, 0.95]).unstack(level=-1)
    q.columns = [f"{prefix}_q{int(p*100):02d}" for p in q.columns]
    return features.join(q, how="left")


def add_pairwise_features(
    interactions: pd.DataFrame,
    user_col: str = "user_id",
    item_col: str = "item_id",
    dt_col: str = "datetime",
    relevance_col: str = "watched_pct",
    cold_user_threshold: int = 1,
    cold_item_threshold: int = 1,
) -> pd.DataFrame:
    """
    Вход: interactions с колонками [user_id, item_id, datetime, total_dur, watched_pct, weight]
    Выход: тот же датафрейм + новые фичи (user- и item-агрегаты примержены обратно на строки взаимодействий).
    """
    df = _ensure_datetime(interactions, dt_col)

    # ---------- Кол-во взаимодействий ----------
    u_cnt = df.groupby(user_col).size().rename("user_interactions_cnt")
    i_cnt = df.groupby(item_col).size().rename("item_interactions_cnt")

    df = df.join(u_cnt, on=user_col).join(i_cnt, on=item_col)

    df["user_log_interactions_cnt"] = _safe_log1p(df["user_interactions_cnt"])
    df["item_log_interactions_cnt"] = _safe_log1p(df["item_interactions_cnt"])

    # ---------- Средний log(кол-ва) "по соседям" ----------
    # item_avg_user_log: средний user_log_interactions_cnt среди пользователей, которые взаимодействовали с item
    item_avg_user_log = (
        df.groupby(item_col)["user_log_interactions_cnt"]
        .mean()
        .rename("item_avg_user_log_interactions_cnt")
    )
    # user_avg_item_log: средний item_log_interactions_cnt среди айтемов, с которыми взаимодействовал user
    user_avg_item_log = (
        df.groupby(user_col)["item_log_interactions_cnt"]
        .mean()
        .rename("user_avg_item_log_interactions_cnt")
    )

    df = df.join(item_avg_user_log, on=item_col).join(user_avg_item_log, on=user_col)

    # разница (как ты просил) между "своим" кол-вом и средним (по соседям)
    df["user_diff_cnt_vs_avg_items"] = df["user_interactions_cnt"] - df["user_avg_item_log_interactions_cnt"].fillna(0.0)
    df["item_diff_cnt_vs_avg_users"] = df["item_interactions_cnt"] - df["item_avg_user_log_interactions_cnt"].fillna(0.0)

    # cold flags
    df["user_cold_flag"] = (df["user_interactions_cnt"] <= cold_user_threshold).astype("int8")
    df["item_cold_flag"] = (df["item_interactions_cnt"] <= cold_item_threshold).astype("int8")

    # ---------- Timestamp-based ----------
    if dt_col in df.columns and np.issubdtype(df[dt_col].dtype, np.datetime64):
        # min/max ts
        u_min = df.groupby(user_col)[dt_col].min().rename("user_min_ts")
        u_max = df.groupby(user_col)[dt_col].max().rename("user_max_ts")
        i_min = df.groupby(item_col)[dt_col].min().rename("item_min_ts")
        i_max = df.groupby(item_col)[dt_col].max().rename("item_max_ts")

        df = (
            df.join(u_min, on=user_col)
              .join(u_max, on=user_col)
              .join(i_min, on=item_col)
              .join(i_max, on=item_col)
        )

        # history length
        df["user_history_timedelta"] = df["user_max_ts"] - df["user_min_ts"]
        df["item_history_timedelta"] = df["item_max_ts"] - df["item_min_ts"]

        df["user_history_days"] = df["user_history_timedelta"].dt.total_seconds() / 86400.0
        df["item_history_days"] = df["item_history_timedelta"].dt.total_seconds() / 86400.0

        # log number of interaction days: кол-во уникальных дней с взаимодействиями
        tmp_date = df[dt_col].dt.floor("D")
        u_days = tmp_date.groupby(df[user_col]).nunique().rename("user_interaction_days_cnt")
        i_days = tmp_date.groupby(df[item_col]).nunique().rename("item_interaction_days_cnt")

        df = df.join(u_days, on=user_col).join(i_days, on=item_col)

        df["user_log_interaction_days_cnt"] = _safe_log1p(df["user_interaction_days_cnt"])
        df["item_log_interaction_days_cnt"] = _safe_log1p(df["item_interaction_days_cnt"])

        # days since last interaction (относительно последней даты в логе)
        last_log_ts = df[dt_col].max()
        df["user_days_since_last_interaction"] = (last_log_ts - df["user_max_ts"]).dt.total_seconds() / 86400.0
        df["item_days_since_last_interaction"] = (last_log_ts - df["item_max_ts"]).dt.total_seconds() / 86400.0

    # ---------- Relevance-based (по watched_pct) ----------
    if relevance_col in df.columns:
        # user relevance stats
        g_u = df.groupby(user_col)[relevance_col]
        u_mean = g_u.mean().rename("user_rel_mean")
        u_std = g_u.std(ddof=0).rename("user_rel_std")

        # item relevance stats
        g_i = df.groupby(item_col)[relevance_col]
        i_mean = g_i.mean().rename("item_rel_mean")
        i_std = g_i.std(ddof=0).rename("item_rel_std")

        df = (
            df.join(u_mean, on=user_col)
              .join(u_std, on=user_col)
              .join(i_mean, on=item_col)
              .join(i_std, on=item_col)
        )

        # quantiles
        # (делаем через отдельные таблицы, чтобы не раздувать расчёты на каждой строке)
        u_q = g_u.quantile([0.05, 0.5, 0.95]).unstack(level=-1)
        u_q.columns = ["user_rel_q05", "user_rel_q50", "user_rel_q95"]

        i_q = g_i.quantile([0.05, 0.5, 0.95]).unstack(level=-1)
        i_q.columns = ["item_rel_q05", "item_rel_q50", "item_rel_q95"]

        df = df.join(u_q, on=user_col).join(i_q, on=item_col)

        # ---------- Abnormality (простой вариант через KL divergence p_u(item) || q(item)) ----------
        # q(item): доля взаимодействий с item во всём логе
        item_pop = (df.groupby(item_col).size() / len(df)).rename("q_item_pop")

        # p_u(item): доля взаимодействий пользователя с item
        ui_cnt = df.groupby([user_col, item_col]).size().rename("ui_cnt").reset_index()
        u_total = ui_cnt.groupby(user_col)["ui_cnt"].sum().rename("u_total").reset_index()

        ui_cnt = ui_cnt.merge(u_total, on=user_col, how="left")
        ui_cnt["p_ui"] = ui_cnt["ui_cnt"] / ui_cnt["u_total"]

        ui_cnt = ui_cnt.merge(item_pop.reset_index(), on=item_col, how="left")

        eps = 1e-12
        ui_cnt["kl_term"] = ui_cnt["p_ui"] * np.log((ui_cnt["p_ui"] + eps) / (ui_cnt["q_item_pop"] + eps))
        user_abn = ui_cnt.groupby(user_col)["kl_term"].sum().rename("user_pref_abnormality_kl")

        df = df.join(user_abn, on=user_col)

    return df

In [ ]:
# Take few models to compare
# To transfer CatBoostRanker we use CatBoostReranker
# We will calculate several classic (precision@k and recall@k) and "beyond accuracy" metrics
metrics = {
    "prec@10": Precision(k=10),
    "recall@10": Recall(k=10),
    "novelty@10": MeanInvUserFreq(k=10),
    # "serendipity@10": Serendipity(k=10),
    "MAP@10": MAP(k=10),
    "NDCG@10": NDCG(k=10),
    "HitRate@10": HitRate(k=10)
}

K_RECS = 10

first_stage = [
    # CandidateGenerator(PopularModel(), num_candidates=100, keep_ranks=True, keep_scores=True),
    # CandidateGenerator(
    #     ImplicitItemKNNWrapperModel(CosineRecommender()),
    #     num_candidates=100,
    #     keep_ranks=True,
    #     keep_scores=True
    # ),
    # CandidateGenerator(
    #     ImplicitALSWrapperModel(
    #       AlternatingLeastSquares(
    #         factors=10,  # latent embeddings size
    #         regularization=0.1,
    #         iterations=10,
    #         alpha=50,  # confidence multiplier for non-zero entries in interactions
    #         random_state=RANDOM_STATE)),
    # num_candidates=100, keep_ranks=True, keep_scores=True),
    CandidateGenerator(
        LightFMWrapperModel(
            LightFM(no_components=10,
                    loss="bpr",
                    random_state=RANDOM_STATE)),
    num_candidates=100, keep_ranks=True, keep_scores=True
)
]


class CustomFeatureCollector(CandidateFeatureCollector):

    def __init__(self, log_df) -> None:
        self.log_df = log_df


    def _get_user_item_features(
        self, useritem: pd.DataFrame, dataset: Dataset, fold_info: tp.Optional[tp.Dict[str, tp.Any]]
    ) -> pd.DataFrame:
        print(fold_info)
        users_without_features = pd.DataFrame(
            np.setdiff1d(dataset.user_id_map.external_ids, self.log_df['user_id'].unique()),
            columns=['user_id']
        )
        # print('users_without_features',users_without_features) #здесь пустой датасет возрвщается потмоу что у меня внешние айдишники в dataset == self.log_df['user_id'].unique() сгенерированому датасету над признакми
        # print(self.log_df)

        test_data = self.log_df[self.log_df['user_id'].isin(useritem['user_id'].values)].fillna(0)
        # print('return ', test_data)
        # return test_data[['item_id','user_id','user_log_interactions_cnt']]
        # return test_data[['item_id','user_id','user_log_interactions_cnt']]

        return test_data[[i for i in test_data.columns if not i in ["datetime","user_min_ts","user_max_ts","item_min_ts","item_max_ts","user_history_timedelta","item_history_timedelta"]]]

splitter = TimeRangeSplitter("10D", n_splits=1)
two_stage_CatBoostReranker = CandidateRankingModel(
    candidate_generators=first_stage,
    splitter=splitter,
    reranker=CatBoostReranker(CatBoostRanker(verbose=False, random_state=RANDOM_STATE)),#pool_kwargs=pool_kwargs
    sampler=PerUserNegativeSampler(n_negatives=3, random_state=RANDOM_STATE), # pass sampler to fix random_state
    # feature_collector=CandidateFeatureCollector(),
    feature_collector=CustomFeatureCollector(log_df= add_pairwise_features(train)),
    verbose=1
)

two_stage_CatBoostReranker.fit(dataset_i_u_features_lfm_train)

reco_CatBoostReranker = two_stage_CatBoostReranker.recommend(
    users=test['user_id'].unique(),
    dataset=dataset_i_u_features_lfm_train,
    k=10,
    filter_viewed=True
)

metric_reco_CatBoostReranker = calc_metrics(metrics, reco_CatBoostReranker, test, train)


# 1) Собираем словари в один датафрейм (строки = разные прогоны/наборы метрик)
final_res_met = pd.DataFrame([
    {"stage": "CatBoostReranker", **metric_reco_CatBoostReranker},
])

# 2) (опционально) сделаем stage индексом для удобства
final_res_met  = final_res_met .set_index("stage")

final_res_met
# models = {
#     "popular": PopularModel(),
#     "cosine_knn": ImplicitItemKNNWrapperModel(CosineRecommender()),
#     'iALS':ImplicitALSWrapperModel(
#           AlternatingLeastSquares(
#             factors=10,  # latent embeddings size
#             regularization=0.1,
#             iterations=10,
#             alpha=50,  # confidence multiplier for non-zero entries in interactions
#             random_state=RANDOM_STATE)),
#     'LightFM':LightFMWrapperModel(
#             LightFM(no_components=10,
#                     loss="bpr",
#                     random_state=RANDOM_STATE)),
#     "two_stage_catboost_ranker": two_stage_catboost_ranker,
# }
# # To transfer CatBoostRanker we use CatBoostReranker




# cv_results = cross_validate(
#     dataset=dataset,
#     splitter=splitter,
#     models=models,
#     metrics=metrics,
#     k=K_RECS,
#     filter_viewed=True,
# )
# We will calculate several classic (precision@k and recall@k) and "beyond accuracy" metrics



In [ ]:
 add_pairwise_features(train)

In [ ]:
rty = two_stage_CatBoostReranker.get_train_with_targets_for_reranker(dataset_train)
rty

In [ ]:
two_stage_CatBoostReranker.reranker.model

In [ ]:
import shap
from catboost import Pool

# сэмплируем для shap_values
X_train_subs, _, y_train_subs, __ = train_test_split(X_train, y_train,
                                                     test_size=0.9,
                                                     random_state=42)
# считаем shap_values
shap_values = two_stage_CatBoostReranker.reranker.model.get_feature_importance(Pool(X_train_subs, y_train_subs,
                                                   cat_features=cat_col), type='ShapValues')

expected_value = shap_values[0, -1]
shap_values = shap_values[:, :-1]
print(X_train.shape,X_train_subs.shape)
plt.title("Важность фичей на train")

shap.summary_plot(
    shap_values,
    X_train_subs
)

In [ ]:
rty[rty['target']==1]

In [ ]:
train[train['user_id']==968969]

In [ ]:
train[train['datetime']>='2021-08-08']

In [ ]:
two_stage_CatBoostReranker.split_to_history_dataset_and_train_targets(dataset_train,splitter)

In [ ]:
train['datetime'].min(),train['datetime'].max()

In [ ]:

metrics = {
    "prec@10": Precision(k=10),
    "recall@10": Recall(k=10),
    "novelty@10": MeanInvUserFreq(k=10),
    # "serendipity@10": Serendipity(k=10),
    "MAP@10": MAP(k=10),
    "NDCG@10": NDCG(k=10),
    "HitRate@10": HitRate(k=10)
}
metric_reco_CatBoostReranker = calc_metrics(metrics, reco_CatBoostReranker, test, train)
# 1) Собираем словари в один датафрейм (строки = разные прогоны/наборы метрик)
final_res_met = pd.DataFrame([
    {"stage": "CatBoostReranker", **metric_reco_CatBoostReranker},
])

# 2) (опционально) сделаем stage индексом для удобства
final_res_met  = final_res_met .set_index("stage")

final_res_met

In [ ]:
# To transfer CatBoostRanker we use CatBoostReranker
splitter = TimeRangeSplitter("7D", n_splits=1)


two_stage_catboost_ranker = CandidateRankingModel(
    candidate_generators=first_stage,
    splitter=splitter,
    reranker=CatBoostReranker(CatBoostRanker(verbose=False, random_state=RANDOM_STATE)),#pool_kwargs=pool_kwargs
    sampler=PerUserNegativeSampler(n_negatives=3, random_state=RANDOM_STATE), # pass sampler to fix random_state
    # feature_collector=CandidateFeatureCollector(),
    feature_collector=CustomFeatureCollector(log_df= add_pairwise_features(interactions)),
    verbose=1
)
cv_results = cross_validate(
    dataset=dataset,
    splitter=splitter,
    models=models,
    metrics=metrics,
    k=K_RECS,
    filter_viewed=True,
)